# Huấn luyện mô hình XGBoost phát hiện gian lận ngân hàng

Notebook production-ready cho **Google Colab**, huấn luyện mô hình phân loại gian lận trên dữ liệu clickstream ngân hàng (~49.851 dòng).

**Pipeline:** Tiền xử lý → Stratified Split → Optuna (PR-AUC) → MLflow tracking → Đánh giá → SHAP → Export JSON cho Flink.

| Bước | Công cụ |
|------|--------|
| Hyperparameter tuning | Optuna |
| Experiment tracking | MLflow (local) |
| Explainability | SHAP |
| Deployment artifact | `xgboost_fraud_model.json` |

## 1. Cài đặt môi trường

Chạy cell này trước tiên trên Colab. Các thư viện được pin phiên bản ổn định để tái lập kết quả.

In [ ]:
!pip install -q xgboost==2.1.4 optuna==4.2.1 mlflow==2.20.1 shap==0.46.0 \
    scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 pandas==2.2.3

## 2. Import thư viện & cấu hình toàn cục

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 3. Tải dữ liệu

Dataset gồm **49.851** sự kiện clickstream với các cột:

- **Features:** `event_type`, `amount`, `ip_address`, `tx_count_1h`, `amount_avg_1h`, `amount_vs_avg`, `hour_of_day`
- **Target:** `is_fraud` (0/1)

Trên Colab, chọn **một** trong các cách sau:
1. Upload file CSV trực tiếp (mặc định: `/content/ml_fraud_features.csv`)
2. Mount Google Drive và trỏ `DATA_PATH` tới file trên Drive
3. Clone repo và dùng file export từ Spark batch pipeline

In [ ]:
# --- Cấu hình đường dẫn dữ liệu (chỉnh theo môi trường Colab của bạn) ---
DATA_PATH = Path("/content/ml_fraud_features.csv")

# Tuỳ chọn: mount Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_PATH = Path("/content/drive/MyDrive/banking_fraud/ml_fraud_features.csv")

# Tuỳ chọn: upload trực tiếp nếu file chưa tồn tại
if not DATA_PATH.exists():
    try:
        from google.colab import files

        print("Chưa tìm thấy CSV. Vui lòng chọn file để upload...")
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        DATA_PATH = Path("/content") / uploaded_name
    except ImportError:
        raise FileNotFoundError(
            f"Không tìm thấy {DATA_PATH}. Đặt file CSV vào đúng path hoặc chạy trên Colab để upload."
        )

df = pd.read_csv(DATA_PATH)
print(f"Kích thước dataset: {df.shape[0]:,} dòng × {df.shape[1]} cột")
display(df.head())
display(df.describe(include="all").T)

### 3.1 Kiểm tra phân phối nhãn

Dữ liệu gian lận thường **mất cân bằng**. Metric chính là **PR-AUC** và **F1** thay vì ROC-AUC.

In [ ]:
TARGET_COL = "is_fraud"
FEATURE_COLS = [
    "event_type",
    "amount",
    "ip_address",
    "tx_count_1h",
    "amount_avg_1h",
    "amount_vs_avg",
    "hour_of_day",
]
CATEGORICAL_OHE = ["event_type"]
CATEGORICAL_FREQ = ["ip_address"]
NUMERIC_COLS = ["amount", "tx_count_1h", "amount_avg_1h", "amount_vs_avg", "hour_of_day"]

missing_cols = set(FEATURE_COLS + [TARGET_COL]) - set(df.columns)
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_cols}")

label_counts = df[TARGET_COL].value_counts().sort_index()
fraud_ratio = df[TARGET_COL].mean()
print(f"Tỷ lệ gian lận (is_fraud=1): {fraud_ratio:.2%}")
print(label_counts.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COL, ax=ax)
ax.set_title("Phân phối nhãn is_fraud")
ax.set_xlabel("is_fraud")
ax.set_ylabel("Số lượng")
plt.tight_layout()
plt.show()

## 4. Tiền xử lý dữ liệu

Chiến lược encoding:
- **`event_type`:** One-Hot Encoding (số lượng category nhỏ, ~7 giá trị)
- **`ip_address`:** Frequency Encoding (tránh sparse matrix ~7.588 chiều nếu One-Hot)

Encoder được **fit chỉ trên tập train** để tránh data leakage.

In [ ]:
@dataclass
class FraudFeaturePreprocessor:
    """Fit trên train, transform train/test/inference với cùng quy tắc encoding."""

    event_type_categories: list[str] | None = None
    ip_frequency_map: dict[str, float] | None = None

    def fit(self, frame: pd.DataFrame) -> "FraudFeaturePreprocessor":
        self.event_type_categories = sorted(frame["event_type"].astype(str).unique().tolist())
        ip_counts = frame["ip_address"].astype(str).value_counts()
        self.ip_frequency_map = ip_counts.to_dict()
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        if self.event_type_categories is None or self.ip_frequency_map is None:
            raise RuntimeError("Gọi fit() trước transform().")

        working = frame.copy()

        # Frequency encoding cho IP (IP mới trong test → frequency = 0)
        working["ip_address_freq"] = (
            working["ip_address"]
            .astype(str)
            .map(self.ip_frequency_map)
            .fillna(0)
            .astype(np.float32)
        )

        # One-Hot Encoding cho event_type (category chưa thấy → cột = 0)
        event_ohe = pd.get_dummies(
            working["event_type"].astype(str),
            prefix="event_type",
            dtype=np.float32,
        )
        for category in self.event_type_categories:
            col_name = f"event_type_{category}"
            if col_name not in event_ohe.columns:
                event_ohe[col_name] = 0.0
        event_ohe = event_ohe[[f"event_type_{c}" for c in self.event_type_categories]]

        numeric_part = working[NUMERIC_COLS].astype(np.float32)
        ip_part = working[["ip_address_freq"]]

        features = pd.concat([numeric_part, ip_part, event_ohe], axis=1)
        features = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return features

    def feature_names(self) -> list[str]:
        if self.event_type_categories is None:
            raise RuntimeError("Preprocessor chưa được fit.")
        return NUMERIC_COLS + ["ip_address_freq"] + [
            f"event_type_{c}" for c in self.event_type_categories
        ]

    def to_artifact(self) -> dict:
        return {
            "event_type_categories": self.event_type_categories,
            "ip_frequency_map": self.ip_frequency_map,
            "numeric_cols": NUMERIC_COLS,
        }


X_raw = df[FEATURE_COLS].copy()
y = df[TARGET_COL].astype(int).values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

preprocessor = FraudFeaturePreprocessor().fit(X_train_raw)
X_train = preprocessor.transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)
feature_names = preprocessor.feature_names()

print(f"Train: {X_train.shape[0]:,} mẫu | Test: {X_test.shape[0]:,} mẫu")
print(f"Số feature sau encoding: {X_train.shape[1]}")
print(f"Tỷ lệ fraud train: {y_train.mean():.2%} | test: {y_test.mean():.2%}")
display(X_train.head())

## 5. Thiết lập MLflow Experiment Tracking

MLflow lưu local trên Colab tại `/content/mlruns`. Artifact (model, plots) được version hoá theo run.

In [ ]:
MLFLOW_DIR = Path("/content/mlruns")
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_DIR.as_uri())
mlflow.set_experiment("banking_fraud_xgboost")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('banking_fraud_xgboost').experiment_id}")

## 6. Hyperparameter Tuning với Optuna

Objective tối đa hoá **PR-AUC** (Average Precision) trên 3-fold Stratified CV — phù hợp dữ liệu mất cân bằng hơn ROC-AUC.

Search space: `max_depth`, `learning_rate`, `n_estimators`, `scale_pos_weight`.

In [ ]:
N_OPTUNA_TRIALS = 40
CV_FOLDS = 3

# Ước lượng scale_pos_weight cơ sở từ tập train
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
base_scale_pos_weight = neg_count / max(pos_count, 1)
print(f"Base scale_pos_weight (neg/pos): {base_scale_pos_weight:.2f}")


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_prob),
    }


def build_xgb_classifier(params: dict) -> xgb.XGBClassifier:
    return xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )


def objective(trial: optuna.Trial) -> float:
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight",
            max(0.5, base_scale_pos_weight * 0.25),
            base_scale_pos_weight * 4.0,
        ),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }

    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    pr_scores: list[float] = []

    for fold_idx, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train)):
        model = build_xgb_classifier(params)
        model.fit(
            X_train.iloc[train_idx],
            y_train[train_idx],
            eval_set=[(X_train.iloc[valid_idx], y_train[valid_idx])],
            verbose=False,
        )
        y_valid_prob = model.predict_proba(X_train.iloc[valid_idx])[:, 1]
        pr_scores.append(average_precision_score(y_train[valid_idx], y_valid_prob))

    mean_pr_auc = float(np.mean(pr_scores))
    trial.set_user_attr("cv_pr_auc_std", float(np.std(pr_scores)))
    return mean_pr_auc


with mlflow.start_run(run_name="optuna_hpo") as hpo_run:
    mlflow.log_param("n_trials", N_OPTUNA_TRIALS)
    mlflow.log_param("cv_folds", CV_FOLDS)
    mlflow.log_param("primary_metric", "pr_auc")

    study = optuna.create_study(direction="maximize", study_name="xgboost_fraud_pr_auc")
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_pr_auc", study.best_value)
    mlflow.log_metric("best_cv_pr_auc_std", study.best_trial.user_attrs.get("cv_pr_auc_std", 0.0))

print("\n=== Kết quả Optuna ===")
print(f"Best CV PR-AUC: {study.best_value:.4f}")
print("Best params:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

## 7. Huấn luyện mô hình cuối & ghi log MLflow

Huấn luyện lại trên **toàn bộ tập train** với hyperparameter tốt nhất từ Optuna, sau đó đánh giá trên hold-out test set.

In [ ]:
final_model = build_xgb_classifier(best_params)
final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_test_prob = final_model.predict_proba(X_test)[:, 1]
test_metrics = compute_metrics(y_test, y_test_prob, threshold=0.5)

print("=== Metrics trên Test Set (threshold=0.5) ===")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name.upper():>10}: {metric_value:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, (y_test_prob >= 0.5).astype(int), digits=4))

with mlflow.start_run(run_name="final_xgboost_champion") as final_run:
    mlflow.log_params(best_params)
    mlflow.log_param("train_size", X_train.shape[0])
    mlflow.log_param("test_size", X_test.shape[0])
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("data_path", str(DATA_PATH))

    for metric_name, metric_value in test_metrics.items():
        mlflow.log_metric(f"test_{metric_name}", metric_value)

    mlflow.xgboost.log_model(
        final_model,
        artifact_path="model",
        input_example=X_train.head(5),
    )

    artifact_path = Path("/content/preprocessor_artifact.json")
    artifact_path.write_text(json.dumps(preprocessor.to_artifact(), indent=2), encoding="utf-8")
    mlflow.log_artifact(str(artifact_path), artifact_path="preprocessing")

    FINAL_RUN_ID = final_run.info.run_id

print(f"\nMLflow Run ID (final): {FINAL_RUN_ID}")

## 8. Đánh giá & Trực quan hoá

### 8.1 Confusion Matrix

Ma trận nhầm lẫn cho thấy trade-off giữa False Positive (báo động sai) và False Negative (bỏ sót gian lận) — quan trọng trong ngân hàng.

In [ ]:
y_test_pred = (y_test_prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix (threshold=0.5)")

sns.heatmap(
    cm,
    annot=True,
    fmt=",d",
    cmap="OrRd",
    xticklabels=["Predicted Legit", "Predicted Fraud"],
    yticklabels=["Actual Legit", "Actual Fraud"],
    ax=axes[1],
)
axes[1].set_title("Confusion Matrix (annotated counts)")
axes[1].set_ylabel("Actual")
axes[1].set_xlabel("Predicted")

plt.tight_layout()
plt.show()

cm_path = Path("/content/confusion_matrix.png")
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(str(cm_path), artifact_path="evaluation")

### 8.2 Precision-Recall Curve

Đường PR phản ánh hiệu suất mô hình trên lớp thiểu số (fraud) tốt hơn ROC curve khi dữ liệu mất cân bằng.

In [ ]:
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_test_prob)
pr_auc = average_precision_score(y_test, y_test_prob)

fig, ax = plt.subplots(figsize=(8, 6))
PrecisionRecallDisplay.from_predictions(y_test, y_test_prob, ax=ax, name="XGBoost")
ax.set_title(f"Precision-Recall Curve (PR-AUC = {pr_auc:.4f})")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

pr_path = Path("/content/precision_recall_curve.png")
fig.savefig(pr_path, dpi=150, bbox_inches="tight")
with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(str(pr_path), artifact_path="evaluation")

## 9. Giải thích mô hình (SHAP)

SHAP Summary Plot phục vụ **tuân thủ quy định tài chính** — giải thích vì sao mô hình gắn nhãn gian lận.

Sampling 2.000 mẫu test để tính SHAP nhanh trên Colab (có thể tăng `SHAP_SAMPLE_SIZE` nếu RAM đủ).

In [ ]:
SHAP_SAMPLE_SIZE = min(2000, X_test.shape[0])
shap_sample_idx = np.random.default_rng(RANDOM_STATE).choice(
    X_test.shape[0], size=SHAP_SAMPLE_SIZE, replace=False
)
X_shap = X_test.iloc[shap_sample_idx]

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

# Bar plot — tầm quan trọng feature trung bình
fig_bar, _ = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.title("SHAP Feature Importance (Bar)")
plt.tight_layout()
plt.show()

shap_bar_path = Path("/content/shap_summary_bar.png")
fig_bar.savefig(shap_bar_path, dpi=150, bbox_inches="tight")

# Dot plot — phân phối impact theo từng feature
fig_dot, _ = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    plot_type="dot",
    show=False,
    max_display=20,
)
plt.title("SHAP Summary (Dot)")
plt.tight_layout()
plt.show()

shap_dot_path = Path("/content/shap_summary_dot.png")
fig_dot.savefig(shap_dot_path, dpi=150, bbox_inches="tight")

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(str(shap_bar_path), artifact_path="shap")
    mlflow.log_artifact(str(shap_dot_path), artifact_path="shap")

## 10. Export mô hình cho Flink Pipeline

Lưu mô hình XGBoost dạng JSON (`xgboost_fraud_model.json`) — định dạng native của XGBoost, tương thích với inference trong PyFlink / ONNX runtime.

Đồng thời export artifact preprocessing để tái tạo feature vector khi serving real-time.

In [ ]:
MODEL_EXPORT_PATH = Path("/content/xgboost_fraud_model.json")
PREPROCESSOR_EXPORT_PATH = Path("/content/preprocessor_artifact.json")
FEATURE_ORDER_PATH = Path("/content/feature_names.json")

final_model.save_model(MODEL_EXPORT_PATH)
PREPROCESSOR_EXPORT_PATH.write_text(json.dumps(preprocessor.to_artifact(), indent=2), encoding="utf-8")
FEATURE_ORDER_PATH.write_text(json.dumps(feature_names, indent=2), encoding="utf-8")

print(f"Model saved: {MODEL_EXPORT_PATH} ({MODEL_EXPORT_PATH.stat().st_size / 1024:.1f} KB)")
print(f"Preprocessor saved: {PREPROCESSOR_EXPORT_PATH}")
print(f"Feature order saved: {FEATURE_ORDER_PATH}")

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(str(MODEL_EXPORT_PATH), artifact_path="deployment")
    mlflow.log_artifact(str(PREPROCESSOR_EXPORT_PATH), artifact_path="deployment")
    mlflow.log_artifact(str(FEATURE_ORDER_PATH), artifact_path="deployment")

# Tải file về máy local (Colab)
try:
    from google.colab import files

    files.download(str(MODEL_EXPORT_PATH))
    files.download(str(PREPROCESSOR_EXPORT_PATH))
except ImportError:
    print("Không phải Colab — bỏ qua auto-download.")

## 11. Sanity check inference (tuỳ chọn)

Load lại model từ JSON và xác nhận score khớp với model in-memory.

In [ ]:
loaded_model = xgb.XGBClassifier()
loaded_model.load_model(MODEL_EXPORT_PATH)

sample_probs_memory = final_model.predict_proba(X_test.head(5))[:, 1]
sample_probs_loaded = loaded_model.predict_proba(X_test.head(5))[:, 1]

comparison = pd.DataFrame(
    {
        "prob_in_memory": sample_probs_memory,
        "prob_from_json": sample_probs_loaded,
        "abs_diff": np.abs(sample_probs_memory - sample_probs_loaded),
    }
)
display(comparison)
assert np.allclose(sample_probs_memory, sample_probs_loaded, atol=1e-6), "Inference mismatch!"
print("Sanity check passed — model JSON load OK.")

---

### Bước tiếp theo (Production)

1. Copy `xgboost_fraud_model.json` + `preprocessor_artifact.json` vào artifact store (S3/GCS) hoặc MLflow Model Registry.
2. Thay placeholder `ChampionModelScorer` trong `streaming_layer/fraud_detector.py` bằng inference XGBoost real-time.
3. Đăng ký model là **Champion** trong MLflow và thiết lập CI/CD retrain từ batch pipeline Spark (`batch_feature_pipeline.py`).